# Asistente de Cumplimiento Normativo para Licitaciones Públicas

Este notebook implementa un sistema RAG (Retrieval-Augmented Generation) que permite consultar documentos de licitaciones públicas chilenas (de momento sólo contamos con una licitación).

## Configuración previa
Antes de ejecutar, configura el entorno.
Si cargaste el ipynb en colab no necesitarás cambiar nada más, sólo tener los documentos en la carpeta local.

 si estás en vscode asegúrate de:
1. Tener un archivo `.env` en la misma carpeta que este notebook (copia `.env.example` y completa tus valores)
2. Tener los documentos `.docx` de la licitación en la misma carpeta
3. Haber instalado las dependencias con la celda de instalación

Ver `README.md` para instrucciones detalladas.

In [16]:
#++Dependencias
!pip install -q langchain langchain-community langchain-openai chromadb pypdf docx2txt
!pip install gradio

In [ ]:
#Configuración de modelos
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Carga de secrets: Colab (primario) o VSCode (secundario)
# Se intenta primero con Colab. Si falla (porque no estamos en Colab),
# se carga automáticamente desde el archivo .env local.
try:
    from google.colab import userdata
    os.environ["GITHUB_TOKEN"]    = userdata.get('GITHUB_TOKEN')
    os.environ["GITHUB_BASE_URL"] = userdata.get('GITHUB_BASE_URL')
    print("  · Secrets cargados desde Google Colab")
except ImportError:
    # No estamos en Colab → cargar desde .env (VSCode / local)
    # Asegúrate de tener un archivo .env basado en .env.example
    from dotenv import load_dotenv
    load_dotenv()
    print("  · Secrets cargados desde archivo .env (modo local)")

# Configuración del modelo de lenguaje
llm = ChatOpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    temperature=0.0
)

# Configuración del modelo de embeddings
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("GITHUB_TOKEN"),
    openai_api_base=os.environ["GITHUB_BASE_URL"],
    check_embedding_ctx_length=False
)

print("Modelos configurados")

Modelos configurados


Testeando y revisando los documentos me di cuenta que hay una tablas que no reconoce en la licitación (archivo original es .pdf y en drive lo descargué como word, ese cambio modifica la estructura, algunos cambios los corrigue el lector de docx, pero otros no), por ende es necesario limpiar y ordenar el documento:
1. RUIDO DE TABLA DE CHECKLIST
       La tabla de evaluación técnica (módulo / funcionalidad / cumple)
       convierte la columna 'si/no' en líneas sueltas que no aportan
       información semántica y generan ~390 tokens de ruido en los
       fragmentos.  Se eliminan.

2. NOMBRES DE MÓDULOS FRAGMENTADOS
       El conversor PDF→DOCX partió algunos encabezados de módulo en dos
       o tres párrafos separados.  docx2txt los une con salto de línea,
       por lo que el texto plano muestra, por ejemplo:
           'MÓDULO DE\nADMINISTRACIÓN DE\nSISTEMAS'
       en vez de 'MÓDULO DE ADMINISTRACIÓN DE SISTEMAS'.
       Se normalizan los 5 casos identificados.

El resto del texto (cuerpo de cada módulo, bases administrativas, criterios de evaluación) no se toca.

In [ ]:
import re
import uuid
from typing import List
from langchain_core.documents import Document
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

def _limpiar_texto_licitacion(texto: str) -> str:
    """
    Corrige dos artefactos que deja la conversión PDF→DOCX sobre este
    archivo específico:
    """
    # Eliminar líneas de marcador si/no
    # Estas líneas provienen de la columna "Cumple" de la tabla de
    # evaluación técnica.  Son siempre líneas aisladas con ese literal.
    texto = re.sub(r'\n[ \t]*si/no[ \t]*\n', '\n', texto, flags=re.IGNORECASE)

    #Unir nombres de módulos fragmentados
    # Cada patrón une exactamente el fragmento conocido; no toca nada más.
    correcciones = [
        # "MÓDULO DE \n ADMINISTRACIÓN DE \n SISTEMAS"
        (r'MÓDULO\s+DE\s*\n+ADMINISTRACIÓN\s+DE\s*\n*SISTEMAS',
         'MÓDULO DE ADMINISTRACIÓN DE SISTEMAS'),
        # "MÓDULO DE DECLARACIÓN \n DE INTERESES Y PATRIMONIO"
        (r'MÓDULO\s+DE\s+DECLARACIÓN\s*\n+DE\s+INTERESES\s+Y\s+PATRIMONIO',
         'MÓDULO DE DECLARACIÓN DE INTERESES Y PATRIMONIO'),
        # "MÓDULO DE \n REPORTABILIDAD"
        (r'MÓDULO\s+DE\s*\n+REPORTABILIDAD',
         'MÓDULO DE REPORTABILIDAD'),
        # "MÓDULO DE AUTO \n ... \n CONSULTA"  (contenido intermedio)
        (r'MÓDULO\s+DE\s+AUTO\s*\n+.*?\n+CONSULTA',
         'MÓDULO DE AUTO CONSULTA'),
        # "MÓDULO DE CAPACITACIÓN participantes."  (primera línea de
        #  contenido pegada al nombre por error del conversor)
        (r'MÓDULO\s+DE\s+CAPACITACIÓN\s+participantes\.',
         'MÓDULO DE CAPACITACIÓN'),
    ]
    for patron, reemplazo in correcciones:
        texto = re.sub(patron, reemplazo, texto, flags=re.DOTALL)

    #Colapsar saltos de línea múltiples resultantes
    texto = re.sub(r'\n{3,}', '\n\n', texto)

    return texto


In [18]:
#Funciones de Carga y fragmentación de documentos
import uuid
from typing import List
from langchain_core.documents import Document
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

#Función para leer archivos .docx.
def cargar_docx(ruta: str, etiqueta: str) -> List[Document]:
    """Carga un .docx y agrega metadatos de origen.
    Usamos .docx porque es el formato estándar en contextos organizacionales
    chilenos (licitaciones, anexos, resoluciones).
    El parámetro 'etiqueta' permite identificar de qué documento
    proviene cada fragmento al momento de responder.
    """
    docs = Docx2txtLoader(ruta).load() ##Lee el archivo .docx y extrae info
    for doc in docs:
        doc.metadata["fuente"] = etiqueta #etiqueta la fuente de los fragmentos
    print(f"  · {etiqueta}: {len(docs)} sección(es) cargada(s)")
    return docs

#Función que fragmenta en chunkx los documentos
def fragmentar(docs: List[Document], chunk_size: int = 800) -> List[Document]:
    """Divide documentos en fragmentos con solapamiento de contexto."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, #chunk de 800 (grande al ser un texto largo)
        chunk_overlap=150, #Solapamiento de info para no perder contexto
        length_function=len,
        add_start_index=True
    )
    return splitter.split_documents(docs)


# Carga de documentos internos
# De momento hay una licitación de prueba y un anexo del postulante a la licitación
# el anexo postulante es una plantilla sin datos, pero tiene la estructura.
# Más adelante podríamos crear una función que lea anexos de los postulantes
# Así tendríamos las bases vs las ofertas
#Los archivos: 1-RESOLUCION_EXENTA_00017_COMPRIMIDO.docx y 2-ANEXO__WORD.docx deben estár en la misma carpeta del notebook
print("Cargando documentos...")
docs_licitacion = cargar_docx("1-RESOLUCION_EXENTA_00017_COMPRIMIDO.docx", "licitacion_base")
docs_anexo      = cargar_docx("2-ANEXO__WORD.docx",                         "plantilla_anexos")

# Fragmentación combinada de todos los documentos cargados
fragmentos = fragmentar(docs_licitacion + docs_anexo)
print(f"\nTotal fragmentos: {len(fragmentos)}")

# Base vectorial de documentos internos
# Chroma almacena los vectores en disco (persist_directory).
# El ID único evita conflictos si se ejecuta el notebook varias veces.
print("\nCreando base vectorial...")
vs = Chroma.from_documents(
    documents=fragmentos,
    embedding=embedding_model,
    persist_directory=f"vs_licitacion_{str(uuid.uuid4())[:6]}"
)

#retriever es el buscador de info vectorial

retriever = vs.as_retriever(
    search_type="mmr", #mmr = Maximum Marginal Relevance : fragmentos relevantes pero diferentes entre si (evita redundancia)
    #k: fragmentos recuperados, fetch_k: fragmentos analizados, lambda_mult: balance relevancia/diversidad (1.0=solo relevancia, 0.0=solo diversidad)
    search_kwargs={"k": 4, "fetch_k": 12, "lambda_mult": 0.7}
)

print("Base vectorial lista")

Cargando documentos...
  · licitacion_base: 1 sección(es) cargada(s)
  · plantilla_anexos: 1 sección(es) cargada(s)

Total fragmentos: 344

Creando base vectorial...
Base vectorial lista


In [19]:
# Función de consulta RAG, ingeniería de prompt
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

#Instrucción base del sistema/modelo, acá determino cómo se comporta
SYSTEM_PROMPT = """
Eres un asistente especializado en licitaciones públicas chilenas.
Responde ÚNICAMENTE basándote en el contexto entregado.
Si la información no está en el contexto, responde:
"No tengo información sobre eso en los documentos de la licitación."
Sé claro, directo y técnico.
"""


def consultar(pregunta: str) -> str:
    """
     Pipeline RAG completo:
    1. Recupera fragmentos relevantes desde la base vectorial (retriever)
    2. Combina los fragmentos en un contexto unificado con su etiqueta de fuente
    3. Construye el prompt con el system prompt + contexto + pregunta del usuario
    4. Envía al LLM y retorna la respuesta como string
    """
    #fragmentos que respondan la pregunta
    fragmentos_relevantes = retriever.invoke(pregunta)
    #formatear los fragmentos como un bloque de contexto único
    # Se incluye la etiqueta de fuente para que el modelo sepa de qué documento proviene
    contexto = "\n---\n".join([
        f"[{d.metadata.get('fuente', '?')}] {d.page_content}"
        for d in fragmentos_relevantes
    ])

    #Creamos las instrucciones del mensaj: prompt del sistema + contexto + pregunta/query
    mensajes = [
        SystemMessage(content=SYSTEM_PROMPT + f"\n\nCONTEXTO:\n{contexto}"),
        HumanMessage(content=pregunta)
    ]

  #Enviamos todo al modelo
    return (llm | StrOutputParser()).invoke(mensajes)


print("Función de consulta lista")

Función de consulta lista


In [20]:
# Estas preguntas son específicas y técnicas → el retriever encuentra contexto relevante
# y el LLM puede responder con precisión.
preguntas_prueba = [
    "¿Cuál es el plazo máximo de implementación permitido y qué pasa si se supera?",
    "¿Qué sistemas del Estado se requiere integrar (SIAPER, PÍAS, MEDIPASS u otros)?",
    "¿Qué documentos debe presentar el oferente para acreditar experiencia previa?",
    "¿Cuánto dura el contrato de arriendo del sistema?",
    "¿Quien eres?",
    "¿En qué puedes ayudarme?",
]

for p in preguntas_prueba:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {p}")
    print(f"{'='*60}")
    print(consultar(p))


PREGUNTA: ¿Cuál es el plazo máximo de implementación permitido y qué pasa si se supera?
El plazo máximo de implementación permitido es de **120 días corridos** desde la fecha del acta de inicio mencionada en las bases administrativas. Si se oferta un plazo superior a este, la oferta será declarada **inadmisible** por no ser conveniente a los intereses del Servicio.

PREGUNTA: ¿Qué sistemas del Estado se requiere integrar (SIAPER, PÍAS, MEDIPASS u otros)?
Según los documentos de la licitación, se requiere la integración del sistema con los siguientes sistemas del Estado:

1. **SIAPER**: Emisión de archivo automático para carga masiva de información.
2. **MEDIPASS**: Integración para la tramitación de licencias médicas emitidas en formato papel y digital.
3. **IMED**: Integración para la tramitación de licencias médicas emitidas en formato papel y digital.
4. **SIGFE 2.0**: Integración en los módulos de administración y gestión.

No se menciona integración con el sistema PÍAS en los doc

In [ ]:
# El sistema debe responder que no tiene información sobre estos temas,
# demostrando que NO alucina ni usa conocimiento externo.
preguntas_prueba = [
    "¿Messi o Cristiano Ronaldo?",
    "¿Nombra 3 Animes?",
]

for p in preguntas_prueba:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {p}")
    print(f"{'='*60}")
    print(consultar(p))


PREGUNTA: ¿Messi o Cristiano Ronaldo?
No tengo información sobre eso en los documentos de la licitación.

PREGUNTA: ¿Nombra 3 Animes?
No tengo información sobre eso en los documentos de la licitación.


In [ ]:
# Preguntas demasiado genéricas → el retriever devuelve fragmentos variados
# y la respuesta puede ser incompleta o superficial.
# Esto demuestra la importancia de formular preguntas específicas.
preguntas_prueba = [
    "¿cuales son los requisitos de la licitacion?",
    "¿Qué se solicita en la licitacion?",
]

for p in preguntas_prueba:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {p}")
    print(f"{'='*60}")
    print(consultar(p))


PREGUNTA: ¿cuales son los requisitos de la licitacion?
Los requisitos de la licitación, según el contexto entregado, son los siguientes:

### Requisitos mínimos para participar:
a. **No haber sido condenado** por alguno de los siguientes delitos:
   - Delitos concursales establecidos en el Título Noveno del Libro Segundo del Código Penal.
   - Delitos establecidos en los numerales 4° párrafos primero, segundo, tercero y quinto; 10° párrafo tercero; 22; 23 párrafo primero; 24 párrafo tercero, y 25 del artículo 97 del Código Tributario.

b. **No haber sido condenado** en virtud de una sentencia firme y ejecutoriada por incumplimiento contractual respecto de un contrato de suministro y prestación de servicios suscrito con organismos sujetos a la ley de compras, derivado de culpa o falta de diligencia en el cumplimiento de sus obligaciones.

### Requisitos y antecedentes legales para ser contratado:
1. **Si el oferente es Persona Natural**:
   - Encontrarse "hábil" en el Registro de Prove

In [ ]:
# Preguntas con contexto y objetivo claro → mejor recuperación y respuesta más útil.
# Demuestran la importancia de la ingeniería de prompts desde el lado del usuario.
preguntas_prueba = [
    "Si yo quisiera postular a esa licitación ¿que información tiene que tener mi proyecto?",
]

for p in preguntas_prueba:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {p}")
    print(f"{'='*60}")
    print(consultar(p))


PREGUNTA: Si yo quisiera postular a esa licitación ¿que información tiene que tener mi proyecto?
Para postular a esta licitación, los proyectos que presentes como experiencia deben cumplir con los siguientes requisitos:

1. **Implementación finalizada**: Los proyectos o experiencias pueden corresponder a contrataciones vigentes, pero el sistema implementado debe estar completamente finalizado al momento de postular a la licitación.

2. **Relación con los módulos requeridos**: Los proyectos o experiencias deben haber incluido la implementación satisfactoria de, al menos, la mitad de los módulos requeridos en el numeral 2 de las bases técnicas de la licitación.

3. **Certificación de implementación satisfactoria**: Debes presentar un "Certificado de Implementación Satisfactoria de Proyectos Similares" (Anexo 2), que incluya:
   - Nombre del proyecto.
   - Servicios prestados.
   - Razón social del cliente.
   - Fecha de realización (periodo: mes/año).
   - Datos de contacto del cliente 